##### 📑 Estudo de Caso: Predição de Demanda com XGBoost
Este notebook documenta o fluxo de criação de um modelo de regressão focado na estimativa de demanda de produtos.

In [94]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
import pickle

In [95]:
df = pd.read_csv("demandas.csv")

In [96]:
df

,data,id_loja,id_produto,categoria,regiao,nivel_estoque,unid_vendidas,unid_pedidas,preco,desconto,condicao_climatica,promocao,preco_concorrente,sazonalidade,epidemia,demanda
0,2022-01-01,S001,P0001,Eletronicos,Norte,195,102,252,72.72,5,Nevada,0,85.73,Inverno,0,115
1,2022-01-01,S001,P0002,Roupas,Norte,117,117,249,80.16,15,Nevada,1,92.02,Inverno,0,229
2,2022-01-01,S001,P0003,Roupas,Norte,247,114,612,62.94,10,Nevada,1,60.08,Inverno,0,157
3,2022-01-01,S001,P0004,Eletronicos,Norte,139,45,102,87.63,10,Nevada,0,85.19,Inverno,0,52
4,2022-01-01,S001,P0005,Alimentos,Norte,152,65,271,54.41,0,Nevada,0,51.63,Inverno,0,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75995,2024-01-30,S005,P0016,Brinquedos,Norte,233,63,0,29.80,5,Nevada,0,32.23,Inverno,0,64
75996,2024-01-30,S005,P0017,Brinquedos,Norte,137,115,141,42.92,5,Nevada,0,40.73,Inverno,0,137
75997,2024-01-30,S005,P0018,Roupas,Norte,197,44,0,17.81,10,Nevada,0,19.41,Inverno,0,68
75998,2024-01-30,S005,P0019,Moveis,Norte,125,58,0,151.72,0,Nevada,0,143.71,Inverno,0,84


Seleção de Features: Definimos as variáveis independentes ($X$) como preço, desconto, estoque e categoria.  Target ($y$): A variável dependente é a demanda, que desejamos prever.  

In [97]:
df.columns

Index(['data', 'id_loja', 'id_produto', 'categoria', 'regiao', 'nivel_estoque',
       'unid_vendidas', 'unid_pedidas', 'preco', 'desconto',
       'condicao_climatica', 'promocao', 'preco_concorrente', 'sazonalidade',
       'epidemia', 'demanda'],
      dtype='str')

In [98]:
features = [
    "preco",
    "desconto",
    "nivel_estoque",
    "promocao",
    "preco_concorrente",
    "categoria"
]

In [99]:
features

['preco',
 'desconto',
 'nivel_estoque',
 'promocao',
 'preco_concorrente',
 'categoria']

In [100]:
tagert = "demanda"

In [101]:
X = df[features].copy()

In [102]:
X

,preco,desconto,nivel_estoque,promocao,preco_concorrente,categoria
0,72.72,5,195,0,85.73,Eletronicos
1,80.16,15,117,1,92.02,Roupas
2,62.94,10,247,1,60.08,Roupas
3,87.63,10,139,0,85.19,Eletronicos
4,54.41,0,152,0,51.63,Alimentos
...,...,...,...,...,...,...
75995,29.80,5,233,0,32.23,Brinquedos
75996,42.92,5,137,0,40.73,Brinquedos
75997,17.81,10,197,0,19.41,Roupas
75998,151.72,0,125,0,143.71,Moveis


In [103]:
y = df[tagert]

In [104]:
y

0        115
1        229
2        157
3         52
4         59
        ... 
75995     64
75996    137
75997     68
75998     84
75999     73
Name: demanda, Length: 76000, dtype: int64

Transformação e seleção de colunas categoricas em ($X$) para formato One-hot-coding (dummies)

In [105]:
categorical_cols = X.select_dtypes(include="object").columns
categorical_cols

C:\Users\AndreFelipeCiccoRiba\AppData\Local\Temp\ipykernel_17520\2973185200.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns


Index(['categoria'], dtype='str')

Utilização do drop_first=True. para eliminar a Multicolinearidade (quando variáveis são cópias ocultas umas das outras), o que torna os modelos matematicamente mais estáveis e leves.

In [106]:
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [107]:
X

,preco,desconto,nivel_estoque,promocao,preco_concorrente,categoria_Brinquedos,categoria_Eletronicos,categoria_Moveis,categoria_Roupas
0,72.72,5,195,0,85.73,False,True,False,False
1,80.16,15,117,1,92.02,False,False,False,True
2,62.94,10,247,1,60.08,False,False,False,True
3,87.63,10,139,0,85.19,False,True,False,False
4,54.41,0,152,0,51.63,False,False,False,False
...,...,...,...,...,...,...,...,...,...
75995,29.80,5,233,0,32.23,True,False,False,False
75996,42.92,5,137,0,40.73,True,False,False,False
75997,17.81,10,197,0,19.41,False,False,False,True
75998,151.72,0,125,0,143.71,False,False,True,False


Divisão 80/20 para representação dos dados

In [108]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)

Definindo o modelo 'xgb' que usará o algoritmo XGBoost para prever a demanda (Regressão). Quero que ele tente minimizar o erro quadrático durante o aprendizado "reg:squarederror".

In [ ]:
xgb = XGBRegressor(objective= "reg:squarederror", n_jobs= -1)

Nesta etapa, saímos do "modelo padrão" e entramos no refinamento técnico. O objetivo é encontrar o equilíbrio entre um modelo que aprende bem e um modelo que não decora os dados (Overfitting).

1. O Dicionário de Estratégias (param_dict)
Cada parâmetro aqui é uma "alavanca" que altera o comportamento matemático do XGBoost:

num_estimators: É o número de árvores de decisão. Mais árvores permitem aprender mais detalhes, mas aumentam o tempo de treino.

max_depth: Controla a altura da árvore. Árvores muito profundas são ótimas para o treino, mas podem falhar em dados novos (variância alta).

learning_rate: É o "passo" do aprendizado. Um valor baixo (ex: 0.01) faz o modelo aprender devagar e com mais precisão, exigindo mais árvores.

subsample & colsample_bytree: Adicionam aleatoriedade ao selecionar frações dos dados e das colunas. Isso torna o modelo mais robusto contra ruídos.

min_child_weight: Um "freio" que impede a criação de ramos na árvore que representem poucos dados, evitando regras específicas demais.

In [ ]:
param_dict = {
    "n_estimators" : [200, 300, 500],
    "max_depth" : [3,4,6,8],
    "learning_rate" : [0.01,0.05,0.1],
    "subsample" : [0.7, 0.8, 1.0],
    "colsample_bytree" : [0.7,0.8,1.0],
    "min_child_weight" : [1,3,5]
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator= xgb,
    param_distributions= param_dict,
    n_iter= 25,
    scoring= "neg_mean_absolute_error",
    cv= 3,
    verbose= 1,
    n_jobs= -1
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
random_search.best_params_

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
y_predict = best_model.predict(X_test)

In [ ]:
rmse =  np.sqrt(mean_squared_error(y_test, y_predict))
rmse

In [ ]:
best_model.feature_importances_

In [ ]:
feature_importance = pd.Series(
    best_model.feature_importances_,
    index= X.columns
).sort_values(ascending=False)

In [ ]:
feature_importance

In [ ]:
feature_importance.plot(kind="bar", title="Features Importantes")

In [ ]:
model_columns = list(X.columns)
with open("model_columns.pkl", "wb") as f:
    pickle.dump(model_columns, f)

In [ ]:
with open("xgboost_demanda_modelo.pkl", "wb") as f:
    pickle.dump(best_model, f)